# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset citation**: Mugotitsa, B., Maina, D., Owoko, H., Akinyi, L., Greenfield, J., Todd, J., Kavu, T. and Kiragga, A. 2026. *A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa*. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

- Entities including record sets, fields, and columns are referenced by their `@id` values for clarity and consistency.

Below, we enumerate the record sets and their fields using `mlcroissant`.

In [ ]:
# List all record sets defined in the dataset
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name','N/A')}")

print("\nSample fields and columns for each RecordSet:")
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']} ({rs.get('name','N/A')})")
    if 'field' in rs:
        for field in rs['field']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name','N/A')}, dataType: {field.get('dataType','N/A')}")
            if 'column' in field:
                for col in field['column']:
                    print(f"    Column @id: {col['@id']}, name: {col.get('name','N/A')}")

## 3. Data Extraction

Load data from selected record set(s) into a pandas DataFrame for analysis. All entities (record sets, fields, columns) are referenced using their `@id` values.

Here, we're extracting and inspecting records from the main survey record set.

In [ ]:
# Example: Use the RecordSet @id for mental health survey records
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Extract records with the RecordSet @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Print columns from the first non-empty RecordSet
for rs_id, df in dataframes.items():
    print(f"Columns in RecordSet {rs_id}: {df.columns.tolist()}")
    print(df.head())
    break

## 4. Exploratory Data Analysis (EDA)

Explore data: filter records based on specific criteria, normalize numeric fields, and group by key attributes.

Below are typical examples, referencing all fields by their `@id`. Adjust parameters as needed based on available fields/columns.

In [ ]:
# Select a numeric field and record set for analysis
# Example: The GAD-7 score field often has an @id like 'https://sen.science/.../gad7_score'
# Use the exact @id from your field overview

# Set the target RecordSet and numeric field @id
target_record_set_id = next(iter(dataframes.keys()))
df = dataframes[target_record_set_id]

# Find numeric columns for demonstration
numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]
if not numeric_fields:
    numeric_fields = [col for col in df.columns if df[col].dtype == 'object' and df[col].str.replace('.','',1).str.isdigit().any()]

# Pick the first available numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using field @id: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in [float, int] else 10
    # Filtering for values above threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field, if available (e.g., gender)
    demo_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower()]
    if demo_fields:
        group_field_id = demo_fields[0]
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization

Visualize distribution of a numeric mental health indicator, grouped by a demographic attribute. All references use `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields and demo_fields:
    numeric_field_id = numeric_fields[0]
    group_field_id = demo_fields[0]

    plt.figure(figsize=(8,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.show()
else:
    print("Suitable numeric and demographic fields not found for visualization.")

## 6. Conclusion

This notebook demonstrated how to load and explore a FAIR² mental health survey dataset using the `mlcroissant` library, referencing all entities by their `@id` identifiers for reproducibility.

With this approach, you can:
- Examine the schema, record sets, and fields
- Extract and filter data using record set and field `@id`
- Normalize and group data for analysis
- Visualize relationships in the data

For deeper analysis, consult the complete list of `@id` values and metadata. Adapt code as needed to reference relevant entities and fields.